In [129]:
# --- Import libraries ---
import os
import json
import pandas as pd
import requests
from typing import List, Dict, Any, Optional
from pathlib import Path
from dotenv import load_dotenv

# Import model-specific libraries
import google.generativeai as genai
from openai import OpenAI
from anthropic import Anthropic

print("All libraries imported successfully!")

All libraries imported successfully!


In [130]:
class LLM:
    """Unified LLM class for processing documents with different AI models."""
    
    # Class variable to track all initialized instances
    _instances = {}
    
    def __init__(self, model_type: str, model_name: str, prompt: str, **kwargs):
        """
        Initialize LLM with specified model and configuration.
        
        Args:
            model_type: Type of model ('gemini', 'claude', 'deepseek', 'openai')
            model_name: Specific model name
            prompt: Prompt template with {{DOCUMENTATION}} placeholder
            **kwargs: Additional model-specific configuration
        """
        self.model_type = model_type.lower()
        self.model_name = model_name
        self.prompt = prompt
        self.config = kwargs
        
        # Register this instance
        LLM._instances[self.model_type] = self
        
        # Initialize model-specific configurations
        self._setup_model()
        
    def _setup_model(self):
        """Setup model-specific configurations and API clients."""
        if self.model_type == 'gemini':
            self._setup_gemini()
        elif self.model_type == 'claude':
            self._setup_claude()
        elif self.model_type == 'deepseek':
            self._setup_deepseek()
        elif self.model_type == 'openai':
            self._setup_openai()
        else:
            raise ValueError(f"Unsupported model type: {self.model_type}")
    
    def _setup_gemini(self):
        """Setup Gemini API configuration."""
        api_key = os.getenv("GOOGLE_API_KEY")
        if not api_key:
            raise ValueError("GOOGLE_API_KEY not found in environment variables")
        
        genai.configure(api_key=api_key)
        
        # Default generation config
        self.generation_config = {
            "max_output_tokens": self.config.get("max_tokens", 2000),
            "temperature": self.config.get("temperature", 0.2),
            "top_p": self.config.get("top_p", 0.95),
            "top_k": self.config.get("top_k", 40)
        }
        
    def _setup_claude(self):
        """Setup Claude API configuration using official Anthropic SDK."""
        api_key = self.config.get("api_key") or os.getenv("ANTHROPIC_API_KEY")
        if not api_key:
            raise ValueError("ANTHROPIC_API_KEY not found")
        
        self.claude_client = Anthropic(api_key=api_key)
        
        # Default configuration
        self.claude_config = {
            "max_tokens": self.config.get("max_tokens", 2000),
            "temperature": self.config.get("temperature", 0.2),
        }
        
    def _setup_deepseek(self):
        """Setup DeepSeek API configuration using OpenAI-compatible interface."""
        api_key = self.config.get("api_key") or os.getenv("DEEPSEEK_API_KEY")
        if not api_key:
            raise ValueError("DEEPSEEK_API_KEY not found")
        
        # DeepSeek uses OpenAI-compatible API with custom base URL
        self.deepseek_client = OpenAI(
            api_key=api_key,
            base_url="https://api.deepseek.com"
        )
        
        # Default configuration
        self.deepseek_config = {
            "max_tokens": self.config.get("max_tokens", 2000),
            "temperature": self.config.get("temperature", 0.2),
            "top_p": self.config.get("top_p", 0.95),
        }
    
    def _setup_openai(self):
        """Setup OpenAI API configuration."""
        api_key = self.config.get("api_key") or os.getenv("OPENAI_API_KEY")
        if not api_key:
            raise ValueError("OPENAI_API_KEY not found")
        
        self.client = OpenAI(api_key=api_key)
        
        # Default configuration
        self.openai_config = {
            "max_tokens": self.config.get("max_tokens", 2000),
            "temperature": self.config.get("temperature", 0.2),
            "top_p": self.config.get("top_p", 0.95),
            "frequency_penalty": self.config.get("frequency_penalty", 0),
            "presence_penalty": self.config.get("presence_penalty", 0)
        }
    
    def query(self, content: str, max_tokens: Optional[int] = None) -> str:
        """
        Query the LLM with given content.
        
        Args:
            content: Text content to process
            max_tokens: Maximum tokens for response (overrides default)
            
        Returns:
            Model response as string
        """
        # Format prompt with content
        formatted_prompt = self.prompt.replace("{{DOCUMENTATION}}", content)
        
        if self.model_type == 'gemini':
            return self._query_gemini(formatted_prompt, max_tokens)
        elif self.model_type == 'claude':
            return self._query_claude(formatted_prompt, max_tokens)
        elif self.model_type == 'deepseek':
            return self._query_deepseek(formatted_prompt, max_tokens)
        elif self.model_type == 'openai':
            return self._query_openai(formatted_prompt, max_tokens)
    
    def _query_gemini(self, prompt: str, max_tokens: Optional[int] = None) -> str:
        """Query Gemini model."""
        try:
            # Update max_tokens if provided
            config = self.generation_config.copy()
            if max_tokens:
                config["max_output_tokens"] = max_tokens
            
            model = genai.GenerativeModel(
                model_name=self.model_name,
                generation_config=config
            )
            
            response = model.generate_content(prompt)
            return response.text
            
        except Exception as e:
            raise Exception(f"Gemini query failed: {str(e)}")
    
    def _query_claude(self, prompt: str, max_tokens: Optional[int] = None) -> str:
        """Query Claude model using official Anthropic SDK."""
        try:
            # Update max_tokens if provided
            config = self.claude_config.copy()
            if max_tokens:
                config["max_tokens"] = max_tokens
            
            response = self.claude_client.messages.create(
                model=self.model_name,
                messages=[{"role": "user", "content": prompt}],
                **config
            )
            
            return response.content[0].text
            
        except Exception as e:
            raise Exception(f"Claude query failed: {str(e)}")
    
    def _query_deepseek(self, prompt: str, max_tokens: Optional[int] = None) -> str:
        """Query DeepSeek model using OpenAI-compatible interface."""
        try:
            # Update max_tokens if provided
            config = self.deepseek_config.copy()
            if max_tokens:
                config["max_tokens"] = max_tokens
            
            response = self.deepseek_client.chat.completions.create(
                model=self.model_name,
                messages=[{"role": "user", "content": prompt}],
                **config
            )
            
            return response.choices[0].message.content
            
        except Exception as e:
            raise Exception(f"DeepSeek query failed: {str(e)}")
    
    def _query_openai(self, prompt: str, max_tokens: Optional[int] = None) -> str:
        """Query OpenAI model."""
        try:
            # Update max_tokens if provided
            config = self.openai_config.copy()
            if max_tokens:
                config["max_tokens"] = max_tokens
            
            response = self.client.chat.completions.create(
                model=self.model_name,
                messages=[{"role": "user", "content": prompt}],
                **config
            )
            
            return response.choices[0].message.content
            
        except Exception as e:
            raise Exception(f"OpenAI query failed: {str(e)}")
    
    def process_json_file(self, json_path: str, output_dir: str, filename: str = None, skip_empty: bool = True) -> None:
        """
        Process JSON file containing preprocessed PDF chunks.
        
        Args:
            json_path: Path to JSON file with document chunks
            output_dir: Directory to save CSV results
            filename: Base filename (without extension) for output CSV. If None, uses model_type + document_number + "_results"
            skip_empty: Whether to skip empty content chunks
        """
        try:
            # Construct output path
            if filename is None:
                json_name = Path(json_path).stem  # e.g., "27" from "27.json"
                filename = f"{self.model_type}_{json_name}_results"
            
            if not filename.endswith('.csv'):
                filename = f"{filename}.csv"
                
            output_path = os.path.join(output_dir, filename)
            
            # Check if output file already exists
            if os.path.exists(output_path):
                print(f"⚠ Output file {filename} already exists. Skipping processing.")
                return
            
            # Ensure output directory exists
            os.makedirs(output_dir, exist_ok=True)
            
            # Load JSON data
            with open(json_path, 'r', encoding='utf-8') as f:
                json_data = json.load(f)
            
            print(f"✓ Processing {Path(json_path).name} -> {filename}")
            print(f"  Loaded JSON with {len(json_data)} chunks")
            
            # Extract document metadata
            doc_metadata = self._extract_document_metadata(json_data)
            print(f"Processing document: {doc_metadata['document_id']} - {doc_metadata['title']}")
            
            results = []
            
            # Process each chunk
            for chunk in json_data:
                doc_id = chunk.get('doc_id', '')
                chunk_num = chunk.get('chunk', 0)
                content = chunk.get('text', '')
                
                # Skip empty content if requested
                if skip_empty and not content.strip():
                    continue
                
                print(f"Processing chunk {chunk_num} of document {doc_id}...")
                
                try:
                    # Query the model
                    model_output = self.query(content)
                    
                    # Try to parse JSON output for success tracking
                    parsed_successfully = self._check_json_parsing(model_output)
                    
                    results.append({
                        "document_id": doc_id,
                        "chunk_number": chunk_num,
                        "model_type": self.model_type,
                        "model_name": self.model_name,
                        "parsed_successfully": parsed_successfully,
                        "raw_output": model_output.strip()
                    })
                    
                except Exception as e:
                    results.append({
                        "document_id": doc_id,
                        "chunk_number": chunk_num,
                        "model_type": self.model_type,
                        "model_name": self.model_name,
                        "parsed_successfully": False,
                        "raw_output": f"Error: {str(e)}"
                    })
            
            # Save results to CSV immediately after processing all chunks
            self._save_results_to_csv(results, output_path)
            print(f"✅ Processing complete. Results saved to {output_path}")
            
        except Exception as e:
            print(f"❌ Error processing JSON file: {e}")
    
    def process_directory(self, directory_path: str, output_dir: Optional[str] = None, 
                         file_pattern: str = "*.json", preview_only: bool = False) -> None:
        """
        Process all JSON files in a directory.
        
        Args:
            directory_path: Path to directory containing JSON files
            output_dir: Output directory (defaults to directory_path/results)
            file_pattern: File pattern to match (default: *.json)
            preview_only: If True, only show what would be processed without actually processing
        """
        directory_path = Path(directory_path)
        
        if output_dir is None:
            output_dir = directory_path / "results"
        else:
            output_dir = Path(output_dir)
        
        # Create output directory
        output_dir.mkdir(exist_ok=True)
        
        # Find matching files
        json_files = list(directory_path.glob(file_pattern))
        
        if not json_files:
            print(f"❌ No files matching {file_pattern} found in {directory_path}")
            return
        
        # Check which files would be processed vs skipped
        files_to_process = []
        files_to_skip = []
        
        for json_file in json_files:
            # Generate expected output filename
            filename = f"{self.model_type}_{json_file.stem}_results.csv"
            output_path = output_dir / filename
            
            if output_path.exists():
                files_to_skip.append((json_file.name, filename))
            else:
                files_to_process.append((json_file.name, filename))
        
        # Display summary
        print(f"\n{'='*60}")
        print(f"📋 PROCESSING PLAN FOR {self.model_type.upper()}")
        print(f"{'='*60}")
        print(f"📁 Input Directory: {directory_path}")
        print(f"💾 Output Directory: {output_dir}")
        print(f"🔍 Pattern: {file_pattern}")
        
        print(f"\n✅ FILES TO PROCESS ({len(files_to_process)}):")
        if files_to_process:
            for json_file, output_file in files_to_process:
                print(f"   {json_file} -> {output_file}")
        else:
            print("   None")
        
        print(f"\n⏭️  FILES TO SKIP ({len(files_to_skip)}) - Already processed:")
        if files_to_skip:
            for json_file, output_file in files_to_skip:
                print(f"   {json_file} -> {output_file} (exists)")
        else:
            print("   None")
        
        print(f"\n📊 SUMMARY:")
        print(f"   Total files found: {len(json_files)}")
        print(f"   Files to process: {len(files_to_process)}")
        print(f"   Files to skip: {len(files_to_skip)}")
        
        if preview_only:
            print(f"\n👀 PREVIEW MODE - No files will be processed")
            return
        
        if not files_to_process:
            print(f"\n✨ All files already processed for {self.model_type}!")
            return
        
        # Ask for confirmation if there are files to process
        estimated_cost = len(files_to_process) * 0.50  # Rough estimate per file
        print(f"\n💰 Estimated cost: ~${estimated_cost:.2f} (rough estimate)")
        
        proceed = input(f"\n❓ Proceed with processing {len(files_to_process)} files? (y/N): ").lower().strip()
        
        if proceed != 'y':
            print("❌ Processing cancelled by user")
            return
        
        print(f"\n🚀 Starting processing...")
        
        # Process files one by one, saving each immediately
        for i, (json_file, output_file) in enumerate(files_to_process, 1):
            print(f"\n📄 Processing file {i}/{len(files_to_process)}: {json_file}...")
            try:
                # Process the file - results are saved immediately within process_json_file
                full_json_path = directory_path / json_file
                self.process_json_file(str(full_json_path), str(output_dir), filename=output_file)
                print(f"✅ Completed {i}/{len(files_to_process)}: {output_file} saved")
            except Exception as e:
                print(f"❌ Failed {json_file}: {str(e)}")
    
        print(f"\n🎉 Batch processing complete! All {len(files_to_process)} files processed and saved individually.")
    
    # Static methods for multi-model operations
    @staticmethod
    def get_working_models():
        """Get all working model instances."""
        working_models = {}
        for name, instance in LLM._instances.items():
            if instance.test_connection():
                working_models[name.title()] = instance
        return working_models
    
    @staticmethod
    def test_all_connections():
        """Test all model connections."""
        print("Testing model connections...\n")
        
        working_models = {}
        for name, instance in LLM._instances.items():
            if instance.test_connection():
                working_models[name.title()] = instance
            print()
        
        print(f"Working models: {list(working_models.keys())}")
        return working_models
    
    @staticmethod
    def process_with_all_models(json_path: str, output_dir: str = "results"):
        """Process a JSON file with all working models."""
        working_models = LLM.get_working_models()
        
        json_path = Path(json_path)
        output_dir = Path(output_dir)
        output_dir.mkdir(exist_ok=True)
        
        print(f"Processing {json_path.name} with all working models...\n")
        
        for name, model in working_models.items():
            # Generate filename for this model
            filename = f"{name.lower()}_{json_path.stem}_results"
            output_path = output_dir / f"{filename}.csv"
            
            # Check if output file already exists
            if output_path.exists():
                print(f"✅ Output file for {name} already exists: {output_path}")
                continue
                
            print(f"📄 Processing with {name}...")
            try:
                # Process and save immediately
                model.process_json_file(str(json_path), str(output_dir), filename)
                print(f"✅ {name} processing complete - saved to {output_path}")
            except Exception as e:
                print(f"❌ {name} processing failed: {str(e)}")
            print()
    
    @staticmethod
    def compare_models_on_file(json_path: str, models_to_compare: List[str] = None, 
                              output_dir: str = "comparison_results"):
        """
        Process the same file with multiple models for comparison.
        
        Args:
            json_path: Path to JSON file
            models_to_compare: List of model names to compare (default: all working models)
            output_dir: Directory to save comparison results
        """
        working_models = LLM.get_working_models()
        
        if models_to_compare is None:
            models_to_compare = list(working_models.keys())
        
        json_path = Path(json_path)
        output_dir = Path(output_dir)
        output_dir.mkdir(exist_ok=True)
        
        print(f"Comparing models {models_to_compare} on {json_path.name}\n")
        
        results = {}
        for model_name in models_to_compare:
            if model_name in working_models:
                # Generate filename for this model
                filename = f"{model_name.lower()}_{json_path.stem}_comparison"
                output_path = output_dir / f"{filename}.csv"
                
                # Check if output file already exists
                if output_path.exists():
                    print(f"✅ Output file for {model_name} already exists: {output_path}")
                    results[model_name] = str(output_path)
                    continue
                    
                print(f"📄 Processing with {model_name}...")
                try:
                    # Process and save immediately
                    working_models[model_name].process_json_file(str(json_path), str(output_dir), filename)
                    results[model_name] = str(output_dir / f"{filename}.csv")
                    print(f"✅ {model_name} complete - saved to {results[model_name]}")
                except Exception as e:
                    print(f"❌ {model_name} failed: {str(e)}")
                    results[model_name] = f"Error: {str(e)}"
            else:
                print(f"⚠ {model_name} not available")
            print()
        
        print("Comparison Results:")
        for model, result in results.items():
            print(f"  {model}: {result}")
        
        return results
    
    @staticmethod
    def quick_test_all_models(test_input: str = None):
        """Quick test of all working models with sample input."""
        working_models = LLM.get_working_models()
        
        if test_input is None:
            test_input = """
            Offshore wind turbines must adhere to Load Resistance Factor Design (LRFD) principles.
            IEC standards currently specify a partial safety factor of 1.35,
            but in hurricane-prone areas of the U.S., API standards require additional robustness checks
            using a 500-year return period for L2 structures.
            """
        
        print("Testing all working models with sample input...\n")
        
        for name, model in working_models.items():
            model.quick_test(test_input)
    
    # Helper methods
    def _extract_document_metadata(self, json_data: List[Dict]) -> Dict[str, Any]:
        """Extract basic document metadata from JSON data."""
        if not json_data:
            return {"document_id": "unknown", "title": "Unknown", "total_chunks": 0}
        
        first_chunk = json_data[0]
        doc_id = first_chunk.get('doc_id', 'unknown')
        
        # Try to extract title from first chunk
        text = first_chunk.get('text', '')
        title = "Unknown"
        
        # Simple heuristic for title extraction
        lines = text.strip().split('\n')
        for line in lines[:10]:
            line = line.strip()
            if line and (line.isupper() or line.startswith('**') or line.startswith('#')):
                title = line.strip('*# ')
                break
        
        return {
            "document_id": doc_id,
            "title": title,
            "total_chunks": len(json_data)
        }
    
    def _check_json_parsing(self, output: str) -> bool:
        """Check if output contains valid JSON structure."""
        try:
            if "```json" in output:
                json_str = output.split("```json")[1].split("```")[0].strip()
                json.loads(json_str)
                return True
            else:
                # Try parsing the entire output as JSON
                json.loads(output)
                return True
        except (json.JSONDecodeError, IndexError):
            return False
    
    def _save_results_to_csv(self, results: List[Dict], output_path: str) -> None:
        """Save processing results to CSV with proper Excel compatibility."""
        df = pd.DataFrame(results)
        # Use Excel-compatible CSV settings
        df.to_csv(
            output_path, 
            index=False, 
            quoting=2,  # Quote all non-numeric fields
            encoding='utf-8-sig',  # UTF-8 with BOM for Excel recognition
            sep=',',  # Ensure comma separator
            lineterminator='\n'  # Standard line terminator
        )

print("LLM class defined successfully!")

LLM class defined successfully!


In [131]:
load_dotenv("D:\OneDrive Files\OneDrive - University of Maryland\PhD research\Python\LLM\API keys\API_keys.env")

True

In [150]:
# --- Import prompts ---
try:
    from prompt_gemini_V1 import prompt as gemini_prompt
    print("✓ Gemini prompt imported")
except ImportError:
    print("⚠ Gemini prompt not found, using default")
    gemini_prompt = "Analyze the following document: {{DOCUMENTATION}}"

try:
    from prompt_claude_lite import prompt as claude_prompt
    print("✓ Claude prompt imported")
except ImportError:
    print("⚠ Claude prompt not found, using default")
    claude_prompt = "Analyze the following document: {{DOCUMENTATION}}"

try:
    from prompt_deepseek_V1 import prompt as deepseek_prompt
    # from prompt_deepseek import prompt as deepseek_prompt
    print("✓ DeepSeek prompt imported")
except ImportError:
    print("⚠ DeepSeek prompt not found, using default")
    deepseek_prompt = "Analyze the following document: {{DOCUMENTATION}}"

try:
    from prompt_gemini_V1 import prompt as openai_prompt
    # from prompt_openai import prompt as openai_prompt
    print("✓ OpenAI prompt imported")
except ImportError:
    print("⚠ OpenAI prompt not found, using default")
    openai_prompt = "Analyze the following document: {{DOCUMENTATION}}"

✓ Gemini prompt imported
✓ Claude prompt imported
✓ DeepSeek prompt imported
✓ OpenAI prompt imported


In [153]:
# --- Initialize LLM instances ---

# Gemini LLM
gemini = LLM(
    model_type="gemini",
    model_name="gemini-1.5-pro",
    prompt=gemini_prompt,
    max_tokens=2000,
    temperature=0.2,
    top_p=0.95,
    top_k=40
)

# Claude LLM (using correct model name)
claude = LLM(
    model_type="claude",
    model_name="claude-3-haiku-20240307",
    # Change to the latest, and more advanced model
    # model_name="claude-3-opus-20240229",
    # model_name='claude-3-5-haiku-20241022',
    prompt=claude_prompt,
    max_tokens=2000,
    temperature=0.2
)

# DeepSeek LLM (using correct model name)
deepseek = LLM(
    model_type="deepseek",
    model_name="deepseek-chat",
    prompt=deepseek_prompt,
    max_tokens=2000,
    temperature=0.2,
    top_p=0.95
)

# OpenAI GPT-4o (corrected model name)
gpt4o = LLM(
    model_type="openai",
    model_name="gpt-4o",
    prompt=openai_prompt,
    max_tokens=2000,
    temperature=0.2,
    top_p=0.95
)

# OpenAI GPT-4o-mini
gpt4o_mini = LLM(
    model_type="openai",
    model_name="gpt-4o-mini",
    prompt=openai_prompt,
    max_tokens=2000,
    temperature=0.2,
    top_p=0.95
)

# OpenAI GPT-4 Turbo
gpt4_turbo = LLM(
    model_type="openai",
    model_name="gpt-4-turbo",
    prompt=openai_prompt,
    max_tokens=2000,
    temperature=0.2,
    top_p=0.95
)

# OpenAI GPT-3.5 Turbo
gpt35_turbo = LLM(
    model_type="openai",
    model_name="gpt-3.5-turbo",
    prompt=openai_prompt,
    max_tokens=2000,
    temperature=0.2,
    top_p=0.95
)

# OpenAI GPT-4.1 Mini
gpt41_mini = LLM(
    model_type="openai",
    model_name="gpt-4.1-mini",
    prompt=openai_prompt,
    max_tokens=2000,
    temperature=0.2,
    top_p=0.95
)

print("All LLM instances created successfully!")
print("Available models: Gemini-1.5-Pro, Claude-3-Haiku, DeepSeek-Chat, GPT-4o, GPT-4o-mini, GPT-4-Turbo, GPT-3.5-Turbo")

All LLM instances created successfully!
Available models: Gemini-1.5-Pro, Claude-3-Haiku, DeepSeek-Chat, GPT-4o, GPT-4o-mini, GPT-4-Turbo, GPT-3.5-Turbo


In [134]:
# # Mac
# json_directory = "/Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/Json_docs"
# output_directory = "/Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/LLM_Results"

# PC - Use raw strings to avoid escape sequence issues
json_directory = r"D:\OneDrive Files\OneDrive - University of Maryland\PhD research\Python\LLM\Json_docs"
output_directory = r"D:\OneDrive Files\OneDrive - University of Maryland\PhD research\Python\LLM\LLM_Results"

# Gemini

In [135]:
# Process Gemini - now with confirmation
gemini.process_directory(json_directory, output_directory)


📋 PROCESSING PLAN FOR GEMINI
📁 Input Directory: D:\OneDrive Files\OneDrive - University of Maryland\PhD research\Python\LLM\Json_docs
💾 Output Directory: D:\OneDrive Files\OneDrive - University of Maryland\PhD research\Python\LLM\LLM_Results
🔍 Pattern: *.json

✅ FILES TO PROCESS (0):
   None

⏭️  FILES TO SKIP (49) - Already processed:
   1.json -> gemini_1_results.csv (exists)
   10.json -> gemini_10_results.csv (exists)
   11.json -> gemini_11_results.csv (exists)
   12.json -> gemini_12_results.csv (exists)
   13.json -> gemini_13_results.csv (exists)
   14.json -> gemini_14_results.csv (exists)
   15.json -> gemini_15_results.csv (exists)
   16.json -> gemini_16_results.csv (exists)
   17.json -> gemini_17_results.csv (exists)
   18.json -> gemini_18_results.csv (exists)
   19.json -> gemini_19_results.csv (exists)
   2.json -> gemini_2_results.csv (exists)
   20.json -> gemini_20_results.csv (exists)
   21.json -> gemini_21_results.csv (exists)
   22.json -> gemini_22_results.csv

# Claude

In [136]:
# claude.process_json_file(
#     json_path=r"D:\OneDrive Files\OneDrive - University of Maryland\PhD research\Python\LLM\Json_docs\18.json",
#     output_dir=r"D:\OneDrive Files\OneDrive - University of Maryland\PhD research\Python\LLM\LLM_Results",
#     filename="claude_18_results"
# )

In [137]:
claude.process_directory(json_directory, output_directory)


📋 PROCESSING PLAN FOR CLAUDE
📁 Input Directory: D:\OneDrive Files\OneDrive - University of Maryland\PhD research\Python\LLM\Json_docs
💾 Output Directory: D:\OneDrive Files\OneDrive - University of Maryland\PhD research\Python\LLM\LLM_Results
🔍 Pattern: *.json

✅ FILES TO PROCESS (0):
   None

⏭️  FILES TO SKIP (49) - Already processed:
   1.json -> claude_1_results.csv (exists)
   10.json -> claude_10_results.csv (exists)
   11.json -> claude_11_results.csv (exists)
   12.json -> claude_12_results.csv (exists)
   13.json -> claude_13_results.csv (exists)
   14.json -> claude_14_results.csv (exists)
   15.json -> claude_15_results.csv (exists)
   16.json -> claude_16_results.csv (exists)
   17.json -> claude_17_results.csv (exists)
   18.json -> claude_18_results.csv (exists)
   19.json -> claude_19_results.csv (exists)
   2.json -> claude_2_results.csv (exists)
   20.json -> claude_20_results.csv (exists)
   21.json -> claude_21_results.csv (exists)
   22.json -> claude_22_results.csv

# OpenAI

In [ ]:
# gpt4o_mini.process_json_file(
#     json_path=r"D:\OneDrive Files\OneDrive - University of Maryland\PhD research\Python\LLM\Json_docs\18.json",
#     output_dir=r"D:\OneDrive Files\OneDrive - University of Maryland\PhD research\Python\LLM\LLM_Results",
#     filename="gpt4o_mini_18_results")

✓ Processing 18.json -> gpt4o_mini_18_results.csv
  Loaded JSON with 25 chunks
Processing document: 18 - 10/24/22** **UNITED STATES DEPARTMENT OF THE INTERIOR ** **Bureau of Ocean Energy Management ** **Office of Renewable Energy Programs ** **DRAFT Information Needed for Issuance of a Notice of Intent (NOI) Under ** **the National Environmental Policy Act (NEPA) for a Construction and ** **Operations Plan (COP)
Processing chunk 1 of document 18...
Processing chunk 2 of document 18...
Processing chunk 3 of document 18...
Processing chunk 4 of document 18...
Processing chunk 5 of document 18...
Processing chunk 6 of document 18...
Processing chunk 7 of document 18...
Processing chunk 8 of document 18...
Processing chunk 9 of document 18...
Processing chunk 10 of document 18...
Processing chunk 11 of document 18...
Processing chunk 12 of document 18...
Processing chunk 13 of document 18...
Processing chunk 14 of document 18...
Processing chunk 15 of document 18...
Processing chunk 16 of 

In [140]:
gpt4o_mini.process_directory(json_directory, output_directory)


📋 PROCESSING PLAN FOR OPENAI
📁 Input Directory: D:\OneDrive Files\OneDrive - University of Maryland\PhD research\Python\LLM\Json_docs
💾 Output Directory: D:\OneDrive Files\OneDrive - University of Maryland\PhD research\Python\LLM\LLM_Results
🔍 Pattern: *.json

✅ FILES TO PROCESS (48):
   1.json -> openai_1_results.csv
   10.json -> openai_10_results.csv
   11.json -> openai_11_results.csv
   12.json -> openai_12_results.csv
   13.json -> openai_13_results.csv
   14.json -> openai_14_results.csv
   15.json -> openai_15_results.csv
   16.json -> openai_16_results.csv
   17.json -> openai_17_results.csv
   18.json -> openai_18_results.csv
   19.json -> openai_19_results.csv
   2.json -> openai_2_results.csv
   20.json -> openai_20_results.csv
   22.json -> openai_22_results.csv
   23.json -> openai_23_results.csv
   24.json -> openai_24_results.csv
   25.json -> openai_25_results.csv
   26.json -> openai_26_results.csv
   27.json -> openai_27_results.csv
   28.json -> openai_28_results.c

# Deepseek

In [154]:
deepseek.process_json_file(
    json_path=r"D:\OneDrive Files\OneDrive - University of Maryland\PhD research\Python\LLM\Json_docs\18.json",
    output_dir=r"D:\OneDrive Files\OneDrive - University of Maryland\PhD research\Python\LLM\LLM_Results",
    filename="deepseek_18_results")

✓ Processing 18.json -> deepseek_18_results.csv
  Loaded JSON with 25 chunks
Processing document: 18 - 10/24/22** **UNITED STATES DEPARTMENT OF THE INTERIOR ** **Bureau of Ocean Energy Management ** **Office of Renewable Energy Programs ** **DRAFT Information Needed for Issuance of a Notice of Intent (NOI) Under ** **the National Environmental Policy Act (NEPA) for a Construction and ** **Operations Plan (COP)
Processing chunk 1 of document 18...
Processing chunk 2 of document 18...
Processing chunk 3 of document 18...
Processing chunk 4 of document 18...
Processing chunk 5 of document 18...
Processing chunk 6 of document 18...
Processing chunk 7 of document 18...
Processing chunk 8 of document 18...
Processing chunk 9 of document 18...
Processing chunk 10 of document 18...
Processing chunk 11 of document 18...
Processing chunk 12 of document 18...
Processing chunk 13 of document 18...
Processing chunk 14 of document 18...
Processing chunk 15 of document 18...
Processing chunk 16 of do